**Last updated: 2026-11-22** - Turned 90 degrees with the rest of the chip: the qubits are a vertical chain, the claws on their west arms, and the feedline runs south to north down the west side, read out from LP_W1.

2026-11-17: - Feedline, resonators and claws only (no qubits, junctions or EPR). Purcell tail too short to draw is now added to the short stub (fixes HFSS PK_ERROR_crossing_edge). HFSS runs in AEDT 2024 R2.

# Readout: Purcell-filter feedline and readout resonators

**To change the design:** edit the numbers in the *Design parameters* cell, then **Restart & Run All**.

The five quarter-wave readout resonators and the shared feedline. Readout is in reflection through launchpad W1. The feedline is itself the Purcell filter, a quarter-wave resonator at the readout band.

Each resonator ends in its claw, drawn where it is on the chip but without the qubit (`readout_claw.py`; ground plane where the qubit would be). The qubits, couplers and control lines are in `qubits_couplers.ipynb`, and the whole chip (with the GDS export) is in `firstpass.ipynb`.

In [ ]:
import importlib
import math
import os

import numpy as np

os.environ.setdefault("QISKIT_METAL_HEADLESS", "True")

from qiskit_metal import designs, Dict
import qiskit_metal as qm
from qiskit_metal.qlibrary.couplers.coupled_line_tee import CoupledLineTee
from qiskit_metal.qlibrary.couplers.line_tee import LineTee
from qiskit_metal.qlibrary.tlines.meandered import RouteMeander
from qiskit_metal.qlibrary.terminations.launchpad_wb import LaunchpadWirebond
from qiskit_metal.qlibrary.terminations.open_to_ground import OpenToGround
from qiskit_metal.qlibrary.terminations.short_to_ground import ShortToGround
from qiskit_metal.qlibrary.tlines.straight_path import RouteStraight
from qiskit_metal.qlibrary.tlines.anchored_path import RouteAnchors

# Local file next to this notebook. A running kernel can miss a file added
# after it started, so refresh the import system's view of the folder first.
importlib.invalidate_caches()
from readout_claw import ReadoutClaw  # local file: readout_claw.py

print("Notebook version: readout.ipynb, last updated 2026-11-22")

## Design parameters

In [ ]:
# =====================================================================
#  DESIGN PARAMETERS
#  Every number you are likely to tune lives in this cell. Edit, then
#  Restart & Run All. The suffix on each name is its unit (_um / _mm).
# =====================================================================

# --- Chip ------------------------------------------------------------
chip_size_mm = 5.0
cpw_width_um = 10  # default centre trace for every route
cpw_gap_um = 6

# --- Readout claws -------------------------------------------------------
# Each resonator ends in a claw around a qubit's west arm. The qubits are
# not drawn here (see readout_claw.py), but the claws sit where they are on
# the chip, so these numbers are the qubits' (from qubits_couplers.ipynb;
# copy them across if they change there). The qubits are a vertical chain at
# orientation 90, Q0 at the south end.
n_resonators = 5
qubit_pitch_mm = 0.75
qubit_column_x_mm = 0.0
cross_etch_arm_length_um = 215.45  # qubit pocket: centre square edge to arm tip
cross_etch_arm_width_um = 89.1  # qubit pocket: width of each arm

# Readout claw on the west arm: a U of metal wrapped around the arm tip.
# Sizes are of the claw metal itself. "Length" runs along the arm,
# "width" across it. The ground spacing (claw gap to cross gap) follows
# from these and the cross sizes above.
claw_outer_length_um = 210.9
claw_outer_width_um = 128.9
claw_inner_length_um = 200
claw_inner_width_um = 107.1
claw_gap_um = 6

# --- Readout --------------------------------------------------------------
# Target bare quarter-wave frequencies, ReadoutRes0..4. The meander lengths are
# computed from these (see the resonator cell), so edit frequencies, not lengths.
resonator_freqs_ghz = [7.3, 7.4, 7.5, 7.6, 7.7]
# HFSS calibration of the resonator lengths. With the lengths computed for
# 7.3-7.7 GHz from (eps_r + 1) / 2 alone, HFSS (full_chip, 2026-11-10) gave
# these frequencies, about 10 % low: the claw and tee capacitances load each
# resonator, and the real effective permittivity differs a little. 1/f_HFSS
# is a straight line in 1/f_design (to 1.3 MHz here), and the resonator
# cell inverts that line to choose each length. Lengths are computed for
# 7.3 -> 6.607 etc., not frequencies the chip had. Replace with new HFSS
# numbers and the lengths they were computed for, to refine.
resonator_hfss_calibration_ghz = {7.3: 6.607, 7.4: 6.684, 7.5: 6.763, 7.6: 6.843, 7.7: 6.921}
substrate_eps_r = 11.45  # silicon at cryogenic temperature
resonator_meander_spacing_um = 50
resonator_lead_um = 150  # straight section at each end of the meander
resonator_fillet_um = 20

# Purcell filter: the shared feedline is itself a quarter-wave resonator at
# the readout band, so readout photons get out and the far-detuned qubits
# are blocked. Read out in reflection from W1. The line starts at its south
# end, where W1 drops onto it and a short stub runs to ground; the stub's
# length sets the filter Q (bandwidth = freq / Q). The line then runs north
# through every tee and ends open past the last one, folding west and back
# south only if it is too long to fit before purcell_fold_y_mm.
purcell_filter_freq_ghz = 7.5
purcell_filter_q = 15  # 500 MHz bandwidth, covering the readout band
# HFSS calibration of the filter length: built for 7.5 GHz from (eps_r + 1)/2,
# the driven S11 run (2026-11-10) put the filter at 7.10 GHz. The tees load
# the line. Update from the eigenmode run's PurcellFilter mode:
# new factor = old factor * f_HFSS / purcell_filter_freq_ghz.
purcell_filter_hfss_factor = 7.10 / 7.5
feedline_x_mm = -1.5
purcell_short_y_mm = -1.80  # the short stub runs south to here, then west into ground
purcell_fold_y_mm = 1.70  # north end of the feedline turns west here...
purcell_fold_x_mm = -1.75  # ...and runs back south at this x to the open end
tee_coupling_length_um = 200
tee_coupling_space_um = 4

# --- Launchpad W1 (readout input) ------------------------------------
w1_pad_y_mm = -1.625
pad_inset_mm = 2.2  # pad centre distance from chip centre

# --- Routing --------------------------------------------------------
feedline_fillet_um = 50

# --- HFSS simulations -------------------------------------------------
# Run in Ansys HFSS through Metal's "hfss" renderer, which starts AEDT if it
# is not already open. Each takes from minutes to an hour or more.
# eigenmode, W1 as a 50 ohm resistor: frequency and Q of every readout and
# of the filter itself, so each readout's kappa
run_hfss_purcell_eigenmode = True
run_hfss_purcell = True  # driven modal: S11 at W1 through the Purcell filter
hfss_purcell_sweep_ghz = (6.5, 8.5, 4001)  # start, stop, points (0.5 MHz steps)
hfss_max_passes = 20
hfss_basis_order = -1  # 0 zero, 1 first, 2 second, -1 mixed
hfss_max_delta_f = 0.1  # eigenmode convergence, % change in frequency per pass
hfss_min_converged = 2  # passes in a row that must meet it before HFSS stops
hfss_max_delta_s = 0.02  # driven-modal convergence, change in S per pass
hfss_buffer_mm = 0.3  # substrate and vacuum beyond what is rendered
hfss_project_file = "readout_hfss.aedt"  # new each run; _2, _3... if taken
hfss_results_dir = "hfss_results/readout"  # mode tables, as CSV

## Readout claws

In [ ]:
design = designs.DesignPlanar()
design.chips.main.size.size_x = f"{chip_size_mm}mm"
design.chips.main.size.size_y = f"{chip_size_mm}mm"
design.variables.cpw_width = f"{cpw_width_um}um"
design.variables.cpw_gap = f"{cpw_gap_um}um"
design.overwrite_enabled = True

pocket_tip_um = cross_etch_arm_width_um / 2 + cross_etch_arm_length_um  # centre to etched arm tip

# Claw metal thicknesses and the ground spacing, from the outer/inner sizes.
claw_side_um = (claw_outer_width_um - claw_inner_width_um) / 2
claw_back_um = claw_outer_length_um - claw_inner_length_um
claw_ground_spacing_um = (claw_inner_width_um - cross_etch_arm_width_um) / 2 - claw_gap_um
if claw_ground_spacing_um < 0:
    print(
        f"WARNING: claw inner width {claw_inner_width_um} um is too narrow for the "
        f"{cross_etch_arm_width_um} um arm pocket plus the claw gap. Ground spacing = "
        f"{claw_ground_spacing_um:.2f} um: the claw overlaps the pocket."
    )
# Distance from qubit centre to the claw finger tips, vs. the E/W arm pocket.
claw_tip_um = pocket_tip_um + claw_ground_spacing_um + claw_gap_um - claw_inner_length_um
if claw_tip_um - claw_gap_um < cross_etch_arm_width_um / 2:
    print(
        f"WARNING: claw fingers reach {claw_tip_um:.1f} um from the qubit centre and "
        f"would run into the qubit's E/W arm pocket (edge at {cross_etch_arm_width_um / 2} um)."
    )


def qubit_y(i):
    """y position (mm) of qubit i (the centre its claw wraps around)."""
    return (i - (n_resonators - 1) / 2) * qubit_pitch_mm


claws = [
    ReadoutClaw(
        design,
        f"Claw{i}",
        options=dict(
            pos_x=f"{qubit_column_x_mm}mm",
            pos_y=f"{qubit_y(i)}mm",
            orientation="90",
            pocket_width=f"{cross_etch_arm_width_um}um",
            pocket_tip=f"{pocket_tip_um}um",
            connection_pads=dict(
                readout=dict(
                    connector_type="0",
                    claw_length=f"{claw_inner_length_um}um",
                    claw_width=f"{claw_side_um}um",
                    claw_width_back=f"{claw_back_um}um",
                    ground_spacing=f"{claw_ground_spacing_um}um",
                    claw_gap=f"{claw_gap_um}um",
                    connector_location="90",
                )
            ),
        ),
    )
    for i in range(n_resonators)
]

print(f"Design initialized on a {chip_size_mm} x {chip_size_mm} mm chip.")
print(f"Placed readout claws Claw0-Claw{n_resonators - 1}.")

## Readout and feedline

In [ ]:
# Each tee is shifted so its coupled-line end sits directly west of the
# claw, keeping every resonator a straight horizontal meander. Tees on
# the south half are mirrored (coupled end on their south side), which keeps
# the column of tees symmetric about the chip centre.
readout_tees = []
for i in range(n_resonators):
    mirror = i < (n_resonators - 1) / 2
    shift_mm = (1 if mirror else -1) * tee_coupling_length_um / 2e3
    readout_tees.append(
        CoupledLineTee(
            design,
            f"CLT{i}",
            options=dict(
                pos_x=f"{feedline_x_mm}mm",
                pos_y=f"{qubit_y(i) + shift_mm}mm",
                orientation="90",
                coupling_length=f"{tee_coupling_length_um}um",
                coupling_space=f"{tee_coupling_space_um}um",
                mirror=mirror,
                # Short the resonator end to ground: with the open claw at
                # the qubit, each readout resonator is a quarter-wave.
                open_termination=False,
            ),
        )
    )

# The readout input launchpad, on the west edge.
LaunchpadWirebond(
    design,
    "LP_W1",
    options=dict(pos_x=f"{-pad_inset_mm}mm", pos_y=f"{w1_pad_y_mm}mm", orientation="0"),
)

print("Placed readout tees and launchpad W1.")

In [ ]:
# Purcell-filter feedline, south to north:
#   ground <- short stub <- W1 input tap -> tees -> (fold) -> open end
# The stub length comes from purcell_filter_q, and the open end's position is
# solved so the whole line (stub included) is a quarter wave at
# purcell_filter_freq_ghz.

c_mm_ghz = 299.792458
eps_eff = (substrate_eps_r + 1) / 2
# Built for the frequency that HFSS puts at purcell_filter_freq_ghz
purcell_design_freq_ghz = purcell_filter_freq_ghz / purcell_filter_hfss_factor
purcell_target_mm = c_mm_ghz / (4 * purcell_design_freq_ghz * eps_eff**0.5)
# Tapped quarter wave: Q = pi / (4 sin^2(pi/2 * tap_distance / length))
tap_from_short_mm = (
    2 * purcell_target_mm / math.pi * math.asin(math.sqrt(math.pi / (4 * purcell_filter_q)))
)


def pin_xy(component, pin):
    return design.components[component].pins[pin].middle


# Input tap butted against the south end of the first tee
tap_half_mm = 0.025
tap_y = pin_xy("CLT0", "prime_start")[1] - tap_half_mm
LineTee(
    design,
    "Purcell_tap",
    options=dict(
        pos_x=f"{feedline_x_mm}mm",
        pos_y=f"{tap_y}mm",
        orientation="270",  # input arm points west; prime_start faces north
        t_length=f"{tap_half_mm * 1e3}um",
    ),
)

_, pad_y = pin_xy("LP_W1", "tie")
feed_in_pins = dict(
    start_pin=dict(component="LP_W1", pin="tie"),
    end_pin=dict(component="Purcell_tap", pin="second_end"),
)
if abs(pad_y - tap_y) < 1e-6:
    RouteStraight(design, "Feed_in", options=dict(pin_inputs=feed_in_pins))
else:  # W1 not level with the tap: jog along halfway in
    mid_x = (feedline_x_mm + pin_xy("LP_W1", "tie")[0]) / 2
    RouteAnchors(
        design,
        "Feed_in",
        options=dict(
            pin_inputs=feed_in_pins,
            anchors=Dict({0: (mid_x, pad_y), 1: (mid_x, tap_y)}),
            fillet=f"{feedline_fillet_um}um",
            advanced=Dict(avoid_collision=False),
        ),
    )

# Short stub: south from the tap to purcell_short_y_mm, then west into ground.
# The trace simply ends in the ground plane at Purcell_short.
stub_mm = tap_from_short_mm - tap_half_mm
short_x = feedline_x_mm - (stub_mm - (tap_y - tap_half_mm - purcell_short_y_mm))
purcell_short = ShortToGround(
    design,
    "Purcell_short",
    options=dict(pos_x=f"{short_x}mm", pos_y=f"{purcell_short_y_mm}mm", orientation="180"),
)
purcell_stub = RouteAnchors(
    design,
    "Purcell_stub",
    options=dict(
        pin_inputs=dict(
            start_pin=dict(component="Purcell_tap", pin="prime_end"),
            end_pin=dict(component="Purcell_short", pin="short"),
        ),
        anchors=Dict({0: (feedline_x_mm, purcell_short_y_mm)}),
        fillet=f"{feedline_fillet_um}um",
        advanced=Dict(avoid_collision=False),
    ),
)
short_x -= stub_mm - purcell_stub.length  # absorb what the fillet takes off
purcell_short.options.pos_x = f"{short_x}mm"
purcell_short.rebuild()
purcell_stub.rebuild()

feed_mids = [
    RouteStraight(
        design,
        f"Feed_mid_{i}",
        options=dict(
            pin_inputs=dict(
                start_pin=dict(component=f"CLT{i}", pin="prime_end"),
                end_pin=dict(component=f"CLT{i + 1}", pin="prime_start"),
            )
        ),
    )
    for i in range(n_resonators - 1)
]

# Everything but the folded tail
fixed_mm = (
    purcell_stub.length
    + 2 * tap_half_mm
    + n_resonators * 2 * tee_coupling_length_um / 1e3
    + sum(f.length for f in feed_mids)
)
tail_mm = purcell_target_mm - fixed_mm

end_y0 = pin_xy(f"CLT{n_resonators - 1}", "prime_end")[1]
fold_run_mm = (purcell_fold_y_mm - end_y0) + (feedline_x_mm - purcell_fold_x_mm)
assert tail_mm > 0, (
    f"The feedline is already {-tail_mm * 1e3:.0f} um longer than a quarter wave at "
    "the last tee: lower purcell_filter_freq_ghz, raise purcell_filter_q (a "
    "shorter stub), or reduce qubit_pitch_mm."
)
# A route shorter than one trace width cannot be drawn: Metal starts and ends
# every route with a lead of half a trace width, so it would double back on
# itself (HFSS then fails with PK_ERROR_crossing_edge).
min_tail_mm = cpw_width_um / 1e3
if tail_mm < min_tail_mm:
    # Tail too short to draw: the line ends open right at the last tee, and
    # the tail's length goes onto the short stub instead. The quarter wave is
    # unchanged; the tap sits tail_mm further from the short (Q a little lower).
    short_x -= tail_mm
    purcell_short.options.pos_x = f"{short_x}mm"
    purcell_short.rebuild()
    purcell_stub.rebuild()
    open_y = end_y0
    purcell_open = OpenToGround(
        design,
        "Purcell_open",
        options=dict(pos_x=f"{feedline_x_mm}mm", pos_y=f"{open_y}mm", orientation="90"),
    )
    purcell_tail = None
elif tail_mm < fold_run_mm:
    # Short tail: the line just runs on north past the last tee and ends open
    open_y = end_y0 + tail_mm
    purcell_open = OpenToGround(
        design,
        "Purcell_open",
        options=dict(pos_x=f"{feedline_x_mm}mm", pos_y=f"{open_y}mm", orientation="90"),
    )
    purcell_tail = RouteStraight(
        design,
        "Purcell_tail",
        options=dict(
            pin_inputs=dict(
                start_pin=dict(component=f"CLT{n_resonators - 1}", pin="prime_end"),
                end_pin=dict(component="Purcell_open", pin="open"),
            )
        ),
    )
else:
    # Long tail: fold west and run back south. The open end is placed, then
    # nudged once to absorb the length the fillets take off the corners.
    open_y = purcell_fold_y_mm - (tail_mm - fold_run_mm)
    purcell_open = OpenToGround(
        design,
        "Purcell_open",
        options=dict(pos_x=f"{purcell_fold_x_mm}mm", pos_y=f"{open_y}mm", orientation="270"),
    )
    purcell_tail = RouteAnchors(
        design,
        "Purcell_tail",
        options=dict(
            pin_inputs=dict(
                start_pin=dict(component=f"CLT{n_resonators - 1}", pin="prime_end"),
                end_pin=dict(component="Purcell_open", pin="open"),
            ),
            anchors=Dict({0: (feedline_x_mm, purcell_fold_y_mm),
                          1: (purcell_fold_x_mm, purcell_fold_y_mm)}),
            fillet=f"{feedline_fillet_um}um",
            advanced=Dict(avoid_collision=False),
        ),
    )
    open_y -= tail_mm - purcell_tail.length
    purcell_open.options.pos_y = f"{open_y}mm"
    purcell_open.rebuild()
    purcell_tail.rebuild()

tail_length_mm = purcell_tail.length if purcell_tail is not None else 0.0
stub_length_mm = purcell_stub.length
purcell_length_mm = (stub_length_mm + 2 * tap_half_mm + n_resonators * 2 * tee_coupling_length_um / 1e3
                     + sum(f.length for f in feed_mids) + tail_length_mm)
tap_mm = stub_length_mm + tap_half_mm  # tap to short, as built
q_built = math.pi / (4 * math.sin(math.pi / 2 * tap_mm / purcell_length_mm) ** 2)
print("Purcell-filter feedline routed (reflection readout from W1).")
tail_note = (f"open tail {tail_length_mm * 1e3:.0f} um" if purcell_tail is not None else
             f"tail of {tail_mm * 1e3:.1f} um too short to draw, added to the short stub")
print(f"  quarter wave {purcell_length_mm:.3f} mm -> {purcell_filter_freq_ghz} GHz in HFSS "
      f"({purcell_design_freq_ghz:.3f} GHz from eps_eff alone); {tail_note}")
print(
    f"  input tap {tap_mm:.3f} mm from the short: Q ~ {q_built:.1f} (target {purcell_filter_q}), "
    f"bandwidth ~ {purcell_filter_freq_ghz / q_built * 1e3:.0f} MHz"
)

In [ ]:
# Quarter-wave length from frequency, with the CPW effective permittivity
# (eps_r + 1) / 2. The resonator also includes the claw's CPW stub and the
# tee's coupled section, so those are taken off the meander's length. The
# claw and tee capacitances pull the real frequency down, so each length is
# computed for a higher design frequency, from the HFSS calibration in the
# parameters: 1/f_HFSS = a / f_design + b. (c_mm_ghz and eps_eff are defined
# in the feedline cell above.)
cal_design = np.array(list(resonator_hfss_calibration_ghz))
cal_hfss = np.array(list(resonator_hfss_calibration_ghz.values()))
cal_a, cal_b = np.polyfit(1 / cal_design, 1 / cal_hfss, 1)
resonator_design_freqs_ghz = [cal_a / (1 / f - cal_b) for f in resonator_freqs_ghz]
claw_stub_mm = claws[0].p.connection_pads.readout.claw_cpw_length
tee_section_mm = readout_tees[0].p.coupling_length + readout_tees[0].p.down_length
resonator_lengths_mm = [
    c_mm_ghz / (4 * f * eps_eff**0.5) - claw_stub_mm - tee_section_mm
    for f in resonator_design_freqs_ghz
]

for i in range(n_resonators):
    RouteMeander(
        design,
        f"ReadoutRes{i}",
        options=dict(
            pin_inputs=dict(
                start_pin=dict(component=f"Claw{i}", pin="readout"),
                end_pin=dict(component=f"CLT{i}", pin="second_end"),
            ),
            total_length=f"{resonator_lengths_mm[i]}mm",
            fillet=f"{resonator_fillet_um}um",
            lead=dict(
                start_straight=f"{resonator_lead_um}um",
                end_straight=f"{resonator_lead_um}um",
            ),
            meander=dict(spacing=f"{resonator_meander_spacing_um}um", asymmetry="0um"),
        ),
    )

print("Staggered meander resonators routed.")
for i, (f, fd, length) in enumerate(
    zip(resonator_freqs_ghz, resonator_design_freqs_ghz, resonator_lengths_mm)
):
    print(f"  ReadoutRes{i}: {f:.3f} GHz in HFSS ({fd:.3f} from eps_eff alone) -> meander {length:.3f} mm")
span_mhz = (max(resonator_freqs_ghz) - min(resonator_freqs_ghz)) * 1e3
print(f"  band {min(resonator_freqs_ghz):.2f}-{max(resonator_freqs_ghz):.2f} GHz, span {span_mhz:.0f} MHz")

In [ ]:
fig = qm.view(design)
fig.suptitle("Readout layout")
fig

## HFSS simulations

Two simulations, each switched on in the parameters cell (`run_hfss_*`). Both render the whole feedline and all five resonators with their claws:

* **Purcell eigenmode.** W1 as a 50 ohm resistor. Every mode's frequency and Q: the filter's own mode (lowest Q) and each readout's linewidth kappa.
* **Purcell readout (driven modal).** A 50 ohm port where W1 drops onto the line. Gives S11 as seen from W1. Each resonator shows up as a phase step and a group-delay peak; the peak's height gives its linewidth.

There are no qubits here, so the qubit-side numbers (dispersive shift, Purcell-limited qubit T1) need the full chip in `firstpass.ipynb`.

Each run of these cells starts a new AEDT project, saved next to this notebook as `hfss_project_file`, or with `_2`, `_3`, ... added if that file exists.

In [ ]:
# Shared by the HFSS cells below: a new AEDT project for this run.
hfss_on = run_hfss_purcell_eigenmode or run_hfss_purcell
hfss_results = {}

if hfss_on:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from qiskit_metal.analyses.simulation import EigenmodeSim, ScatteringImpedanceSim

    # A new AEDT project for this run, saved at once under an unused name.
    # The renderer then attaches to it by name before every simulation, so
    # the designs cannot land in whichever project happens to be active.
    stem, ext = os.path.splitext(os.path.abspath(hfss_project_file))
    hfss_project_path, k = stem + ext, 2
    while os.path.exists(hfss_project_path):
        hfss_project_path, k = f"{stem}_{k}{ext}", k + 1
    hfss = design.renderers.hfss
    # AEDT 2024 R2. Started only if no 2024 R2 copy is open already: the
    # renderer attaches to an AEDT that is already running, so starting
    # another (or a different version) just leaves extra windows, and several
    # open copies can stall the connection.
    import subprocess

    running = subprocess.run(
        ["powershell", "-NoProfile", "-Command",
         "(Get-Process ansysedt -ErrorAction SilentlyContinue).Path"],
        capture_output=True, text=True,
    ).stdout
    if "v242" not in running:
        design.renderers.hfss.open_ansys(path_var="ANSYSEM_ROOT242")
    hfss.start()
    hfss.rdesktop.new_project().save(hfss_project_path)
    hfss.options.project_name = os.path.splitext(os.path.basename(hfss_project_path))[0]
    hfss.options.project_path = None
    hfss.start()  # reconnect, now to the new project
    print(f"HFSS project: {hfss_project_path} (AEDT {hfss.rdesktop.get_version()})")



def hfss_buffer(sim):
    sim.renderer.options.x_buffer_width_mm = hfss_buffer_mm
    sim.renderer.options.y_buffer_width_mm = hfss_buffer_mm


# The parts every simulation renders: the whole feedline, the resonators and
# their claws.
feedline_parts = ["Feed_in", "Purcell_tap", "Purcell_stub", "Purcell_short",
                  "Purcell_tail", "Purcell_open"]
feedline_parts = [part for part in feedline_parts if part in design.components]  # no tail if too short
feedline_parts += [f"CLT{i}" for i in range(n_resonators)]
feedline_parts += [f"Feed_mid_{i}" for i in range(n_resonators - 1)]
readout_parts = [part for i in range(n_resonators) for part in (f"ReadoutRes{i}", f"Claw{i}")]

In [ ]:
# Purcell eigenmode: the feedline and every readout, with W1 as a 50 ohm
# resistor (an RLC boundary where the pad's trace starts). Each mode's Q is
# then set by how much of it leaks out through W1:
#   - the filter: the lowest-Q mode (Q ~ purcell_filter_q)
#   - each readout: kappa / 2 pi = f / Q
# The modes are named by frequency and Q alone.
if run_hfss_purcell_eigenmode:
    pe = EigenmodeSim(design, "hfss")
    hfss_buffer(pe)
    pe.setup.update(
        n_modes=n_resonators + 2,  # readouts, filter, one spare
        min_freq_ghz=0.8 * min(resonator_freqs_ghz),
        max_passes=hfss_max_passes,
        min_converged=hfss_min_converged,
        max_delta_f=hfss_max_delta_f,
        basis_order=hfss_basis_order,
    )
    pe.run(
        name="Purcell_eigenmode",
        components=feedline_parts + readout_parts,
        open_terminations=[],
        port_list=[("Feed_in", "start", 50)],
    )
    pe.plot_convergences()

    freqs, kappas = pe.renderer.pinfo.setup.get_solutions().eigenmodes()
    f_ghz = np.asarray(freqs, dtype=float)
    k_ghz = np.asarray(kappas if kappas is not None else [np.nan] * len(f_ghz), dtype=float)
    if f_ghz.max() > 1e3:  # exported in Hz
        f_ghz, k_ghz = f_ghz / 1e9, k_ghz / 1e9
    q = f_ghz / k_ghz

    # Names: the lowest-Q mode is the filter; the rest are the readouts in
    # frequency order, then anything extra.
    order = list(np.argsort(f_ghz))
    filt = min(order, key=lambda m: q[m])
    labels = {filt: "PurcellFilter"}
    others = [m for m in order if m != filt]
    for i, m in enumerate(others):
        labels[m] = f"ReadoutRes{i}" if i < n_resonators else f"extra{i - n_resonators}"

    targets = {f"ReadoutRes{i}": f for i, f in enumerate(resonator_freqs_ghz)}
    targets["PurcellFilter"] = purcell_filter_freq_ghz
    purcell_modes = pd.DataFrame(
        {
            labels[m]: {
                "f (GHz)": f_ghz[m],
                "target (GHz)": targets.get(labels[m], np.nan),
                "Q": q[m],
                "kappa/2pi (MHz)": k_ghz[m] * 1e3,
            }
            for m in order
        }
    ).T
    os.makedirs(hfss_results_dir, exist_ok=True)
    purcell_modes.to_csv(os.path.join(hfss_results_dir, "Purcell_eigenmode_modes.csv"))
    hfss_results["Purcell_eigenmode"] = purcell_modes
    print("Purcell eigenmode, N1 as 50 ohm:")
    print(purcell_modes.to_string(float_format=lambda v: f"{v:.4g}"))

In [ ]:
# Purcell readout: the full feedline (input tap, short stub, tees, tail and
# open end) and all resonators with their claws. LP_W1 is left out so the port goes
# where the pad's trace starts. The filter is the broad phase wind of 2 pi
# (group delay 4 / kappa_filter). The readouts are steps about a MHz wide,
# so trust the eigenmode cell above for their kappa: here the mesh is only
# refined until S11 stops changing, and |S11| is 1 at every frequency (no
# loss), so the resonators can stay under-resolved.
if run_hfss_purcell:
    drv = ScatteringImpedanceSim(design, "hfss")
    hfss_buffer(drv)
    start_ghz, stop_ghz, count = hfss_purcell_sweep_ghz
    drv.setup.update(
        freq_ghz=purcell_filter_freq_ghz,
        max_passes=hfss_max_passes,
        max_delta_s=hfss_max_delta_s,
        min_converged=hfss_min_converged,
        basis_order=hfss_basis_order,
    )
    drv.setup.sweep_setup.update(
        start_ghz=start_ghz, stop_ghz=stop_ghz, count=count, step_ghz=None, type="Fast"
    )
    drv.run(
        name="Purcell_readout",
        components=feedline_parts + readout_parts,
        open_terminations=[],
        port_list=[("Feed_in", "start", 50)],
    )

    _, _, s_params = drv.renderer.get_params(["S11"])
    f_ghz = np.asarray(s_params.index, dtype=float)
    s11 = np.asarray(s_params["S11"], dtype=complex)
    phase = np.unwrap(np.angle(s11))
    # Group delay, -d(phase)/d(omega). On resonance, a reflection off a
    # resonator peaks at 4 / kappa, so kappa / 2 pi = 2 / (pi * peak delay).
    delay_ns = -np.gradient(phase, 2 * np.pi * f_ghz)

    from scipy.signal import find_peaks

    peaks, _ = find_peaks(delay_ns, prominence=0.2 * delay_ns.max())
    print("Resonances seen from W1 (group-delay peaks):")
    for p in peaks:
        kappa_mhz = 2 / (np.pi * delay_ns[p]) * 1e3
        print(f"  {f_ghz[p]:.4f} GHz   delay {delay_ns[p]:7.1f} ns   kappa/2pi ~ {kappa_mhz:.2f} MHz")
    print(f"Targets: resonators {resonator_freqs_ghz} GHz, "
          f"filter {purcell_filter_freq_ghz} GHz (Q ~ {purcell_filter_q})")

    fig, axs = plt.subplots(3, 1, figsize=(9, 8), sharex=True)
    axs[0].plot(f_ghz, 20 * np.log10(np.abs(s11)))
    axs[0].set_ylabel("|S11| (dB)")
    axs[1].plot(f_ghz, phase)
    axs[1].set_ylabel("phase (rad)")
    axs[2].plot(f_ghz, delay_ns)
    axs[2].plot(f_ghz[peaks], delay_ns[peaks], "x")
    axs[2].set_ylabel("group delay (ns)")
    axs[2].set_xlabel("frequency (GHz)")
    for f in resonator_freqs_ghz:
        for ax in axs:
            ax.axvline(f, color="grey", lw=0.5, ls=":")
    fig.suptitle("S11 at W1 (dotted: target resonator frequencies)")
    fig.tight_layout()
    hfss_results["Purcell_readout"] = dict(f_ghz=f_ghz, s11=s11, peaks_ghz=f_ghz[peaks])

In [ ]:
# Save the AEDT project (every simulation above is a design in it) and
# release the connection to AEDT. AEDT itself stays open for inspection.
if hfss_on:
    hfss.pinfo.project.save()
    print(f"Saved the HFSS project to {hfss_project_path}")
    hfss.stop()